# Laboratorio Evaluativo: Decisión Médica de Vida o Muerte
## Diagnóstico de Cáncer de Mama

¡Es hora de poner a competir a los algoritmos en el mundo real! Usted ha sido contratado por la junta médica de un Hospital Oncológico en Wisconsin.

### **Contexto de Negocio (Medicina)**
El hospital cuenta con una base de datos histórica de biopsias celulares extraídas de pacientes.
El objetivo es construir y entrenar 3 modelos distintos (k-NN, Árboles y Naive Bayes) para predecir si un nuevo tumor es **Maligno (1)** o **Benigno (0)** basándose en las medidas físicas de la célula extraída.

Usted deberá evaluar los 3 modelos, pero **cuidado**: En medicina, cometer un error de Falso Positivo (asustar a una paciente sana) es feo, pero cometer un **Falso Negativo (decirle a una paciente con cáncer maligno que está completamente sana)** es letal.

Su labor es encontrar el modelo que minimice los Falsos Negativos (es decir, que maximice el *Recall* o *Sensibilidad* de la clase maligna).

## **Sección 1: Extracción, Documentación y Estandarización**

### Diccionario Médico de Datos (Las 10 características base)
Para que usted como Científico de Datos entienda la biología de la que estamos hablando, los doctores midieron 10 aspectos del núcleo de las células. Para cada aspecto calcularon 3 cosas: la media (mean), el error estándar (error) y el peor de los casos (worst), dándonos 30 columnas en total.

- **Radio (Radius):** Distancia del centro a los bordes.
- **Textura (Texture):** Variación en la escala de grises de la imagen celular.
- **Perímetro (Perimeter):** Tamaño del borde del núcleo.
- **Área (Area):** Tamaño total de la célula.
- **Suavidad (Smoothness):** Variación local en la longitud de los radios.
- **Compacidad (Compactness):** (Perímetro^2 / Área) - 1.0
- **Concavidad (Concavity):** Gravedad de las porciones cóncavas del contorno.
- **Puntos Cóncavos (Concave points):** Número de partes hundidas en el contorno.
- **Simetría (Symmetry):** Qué tan simétrica es la célula.
- **Dimensión Fractal (Fractal dimension):** Aproximación de la "rugosidad" del borde.

In [1]:
import pandas as pd
import numpy as np
from sklearn.datasets import load_breast_cancer

# Extracción de datos
datos_cancer = load_breast_cancer()
X = pd.DataFrame(datos_cancer.data, columns=datos_cancer.feature_names)

# TRADUCCIÓN AUTOMÁTICA DE COLUMNAS AL ESPAÑOL:
X.columns = [
    col.replace('mean', 'media')
       .replace('worst', 'peor')
       .replace('radius', 'radio')
       .replace('texture', 'textura')
       .replace('perimeter', 'perimetro')
       .replace('area', 'area')
       .replace('smoothness', 'suavidad')
       .replace('compactness', 'compacidad')
       .replace('concave points', 'puntos_concavos')
       .replace('concavity', 'concavidad')
       .replace('symmetry', 'simetria')
       .replace('fractal dimension', 'dimension_fractal')
       .replace(' ', '_')
    for col in X.columns
]

# En sklearn original, 0 = Malignant, 1 = Benign.
# Vamos a invertirlo para que 1 = Malignant (es más intuitivo detectar el peligro como 1)
y = pd.Series(datos_cancer.target).map({0: 1, 1: 0})

display(X.head())

,media_radio,media_textura,media_perimetro,media_area,media_suavidad,media_compacidad,media_concavidad,media_puntos_concavos,media_simetria,media_dimension_fractal,...,peor_radio,peor_textura,peor_perimetro,peor_area,peor_suavidad,peor_compacidad,peor_concavidad,peor_puntos_concavos,peor_simetria,peor_dimension_fractal
0,17.99,10.38,122.80,1001.0,0.11840,0.27760,0.3001,0.14710,0.2419,0.07871,...,25.38,17.33,184.60,2019.0,0.1622,0.6656,0.7119,0.2654,0.4601,0.11890
1,20.57,17.77,132.90,1326.0,0.08474,0.07864,0.0869,0.07017,0.1812,0.05667,...,24.99,23.41,158.80,1956.0,0.1238,0.1866,0.2416,0.1860,0.2750,0.08902
2,19.69,21.25,130.00,1203.0,0.10960,0.15990,0.1974,0.12790,0.2069,0.05999,...,23.57,25.53,152.50,1709.0,0.1444,0.4245,0.4504,0.2430,0.3613,0.08758
3,11.42,20.38,77.58,386.1,0.14250,0.28390,0.2414,0.10520,0.2597,0.09744,...,14.91,26.50,98.87,567.7,0.2098,0.8663,0.6869,0.2575,0.6638,0.17300
4,20.29,14.34,135.10,1297.0,0.10030,0.13280,0.1980,0.10430,0.1809,0.05883,...,22.54,16.67,152.20,1575.0,0.1374,0.2050,0.4000,0.1625,0.2364,0.07678


### **Requerimiento 1: Separar y Escalar**
Como utilizaremos k-NN en nuestro taller comparativo, usted **debe** escalar los datos obligatoriamente.

In [9]:
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler

# TODO: Ejecute el Flujo de Machine Learning

# 1. train_test_split (utilice test_size=0.20, random_state=123)
X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.20, random_state=123, stratify=y
)

# 2. Estandarizar con StandardScaler()
scaler = StandardScaler()
X_train_esc = scaler.fit_transform(X_train)
X_test_esc = scaler.transform(X_test)


---
## **Sección 2: Torneo de Algoritmos**

Entrene a los 3 modelos simultáneamente usando los datos de entrenamiento escalados.

In [10]:
from sklearn.neighbors import KNeighborsClassifier
from sklearn.tree import DecisionTreeClassifier
from sklearn.naive_bayes import GaussianNB
from sklearn.metrics import recall_score, confusion_matrix, classification_report

# TODO: Instancie y Entrene los 3 modelos

# 1. k-NN (n_neighbors=5)
modelo_knn = KNeighborsClassifier(n_neighbors=5)
modelo_knn.fit(X_train_esc, y_train)
pred_knn = modelo_knn.predict(X_test_esc)

# 2. Árbol de Decisión (max_depth=4, random_state=42)
modelo_tree = DecisionTreeClassifier(max_depth=4, random_state=42)
modelo_tree.fit(X_train, y_train)
pred_tree = modelo_tree.predict(X_test)

# 3. Naive Bayes (GaussianNB)
modelo_bayes = GaussianNB()
modelo_bayes.fit(X_train, y_train)
pred_bayes = modelo_bayes.predict(X_test)

---
## **Sección 3: Evaluación Médica (El Veredicto Final)**

Imprima la Matriz de Confusión y el `classification_report` para cada uno de los 3 modelos.

In [12]:
# TODO: Utilice print() y confusion_matrix() para mostrar los resultados de KNN, Árbol y Bayes.
# Observe detalladamente la métrica 'recall' para la clase 1 (Maligno).

print("= k-NN =")
print("Matriz de Confusión:\n", confusion_matrix(y_test, pred_knn))
print("\nReporte de Clasificación:\n", classification_report(y_test, pred_knn))

print("\n= Árbol de Decisión =")
print("Matriz de Confusión:\n", confusion_matrix(y_test, pred_tree))
print("\nReporte de Clasificación:\n", classification_report(y_test, pred_tree))

print("\nNaive Bayes")
print("Matriz de Confusión:\n", confusion_matrix(y_test, pred_bayes))
print("\nReporte de Clasificación:\n", classification_report(y_test, pred_bayes))

= k-NN =
Matriz de Confusión:
 [[70  2]
 [ 2 40]]

Reporte de Clasificación:
               precision    recall  f1-score   support

           0       0.97      0.97      0.97        72
           1       0.95      0.95      0.95        42

    accuracy                           0.96       114
   macro avg       0.96      0.96      0.96       114
weighted avg       0.96      0.96      0.96       114


= Árbol de Decisión =
Matriz de Confusión:
 [[63  9]
 [ 1 41]]

Reporte de Clasificación:
               precision    recall  f1-score   support

           0       0.98      0.88      0.93        72
           1       0.82      0.98      0.89        42

    accuracy                           0.91       114
   macro avg       0.90      0.93      0.91       114
weighted avg       0.92      0.91      0.91       114


Naive Bayes
Matriz de Confusión:
 [[65  7]
 [ 2 40]]

Reporte de Clasificación:
               precision    recall  f1-score   support

           0       0.97      0.90      

### **Sección 4: Decisión de la Junta Médica (Responda Aquí)**
Haga doble clic en esta celda y conteste las preguntas para entregar su recomendación médica al director del hospital.

Recuerde la regla de oro de la medicina oncológica: **Un Falso Negativo (Cáncer no detectado) es inaceptable.** Queremos que el modelo sea hipersensible al peligro.

**Pregunta de Directorio:** Basándose puramente en la métrica *Recall (Sensibilidad)* de la clase 1 (Tumor Maligno) arrojada por los 3 modelos y leyendo los Falsos Negativos en las matrices de confusión:
¿Qué algoritmo demostró ser el mejor y más seguro para detectar tumores malignos? ¿Por qué descartaría a los otros dos a pesar de que pudieran tener mejor *Accuracy* general?
> *[Escriba su respuesta analítica aquí...]*
El mejor modelo fue k-NN porque obtuvo el mayor Recal para la clase 1 y fue el que menos Falsos Negativos cometió en la matriz de confusión
descartaría a los otros dos modelos porque aunque su exactitud global pueda ser alta en este caso no sirve de nada si fallan justo en un caso de cáncer. Prefiero un modelo que sea un poco más alarmista antes que cometer el error de dejar ir a una paciente con cáncer maligno por un Falso Negativo.